# 00 — Environment Check

Verifies that all required packages are importable and the results directory is writable.
Run this notebook before any benchmark run to confirm the environment is healthy.

This notebook is executed by Papermill as part of the Dagster `environment_check` asset.

## Parameters

The cell below is tagged `parameters` so Papermill can inject values at execution time.

In [ ]:
# Injected by Papermill
results_path = None  # Path to results base dir; None → auto-detect

## 1. Package versions

In [ ]:
import sys
import importlib

required_packages = [
    "benchlib",
    "celery",
    "kombu",
    "opentelemetry",
    "pandas",
    "pyarrow",
]

results = {}
for pkg in required_packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'unknown')
        results[pkg] = version
        print(f"  OK  {pkg:<30} {version}")
    except ImportError as e:
        results[pkg] = f"MISSING: {e}"
        print(f"  FAIL {pkg}: {e}")

print(f"\nPython: {sys.version}")

failed = [k for k, v in results.items() if v.startswith('MISSING')]
assert not failed, f"Missing packages: {failed}"

## 2. Results directory

In [ ]:
import os
import tempfile
from pathlib import Path

if results_path is not None:
    base = Path(results_path)
else:
    # Heuristic: look for results/ relative to this notebook's parent
    base = Path(os.environ.get('BENCH_RESULTS_PATH', Path.cwd() / 'results'))

base.mkdir(parents=True, exist_ok=True)
print(f"Results base path: {base}")
print(f"  Exists:   {base.exists()}")
print(f"  Is dir:   {base.is_dir()}")

with tempfile.NamedTemporaryFile(dir=base, prefix='.write_test_', delete=True):
    pass
print("  Writable: True")
print("\nEnvironment check PASSED")

## 3. benchlib smoke import

In [ ]:
from benchlib.run_spec import SMOKE_SPEC
from benchlib.results import ResultDir
from benchlib.producer import BenchmarkProducer

print(f"SMOKE_SPEC.broker    = {SMOKE_SPEC.broker}")
print(f"SMOKE_SPEC.task      = {SMOKE_SPEC.task}")
print(f"SMOKE_SPEC.mode      = {SMOKE_SPEC.mode}")
print("benchlib imports OK")